# Notebook 23 — Predictive Constraint Routing

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 22 forecasted decompression before fallback occurs.

Notebook 23 uses decompression forecasts to route before fallback occurs.

Constraint view:
> predictive routing is useful when forecast risk can stabilize compressed routes before fallback becomes necessary.

## Goals

1. Load Notebook 22 predictive decompression outputs when available.
2. Build reactive and predictive routing policies.
3. Compare fallback, reroute, watch, and accepted states.
4. Estimate stability gain from forecast-informed routing.
5. Export CSV, JSON, Markdown report, and PNG figures.
6. Generate a Colab-downloadable output zip.

This notebook follows the prior repo notebook style:

- separate notebook sections,
- saved figures plus `plt.show()` rendering,
- generated outputs section in the report,
- `figures/` link style in the report.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 22 outputs

Uses:

```text
results/notebook22_predictive_decompression.csv
```

If that file is unavailable, this notebook creates a fallback synthetic compressed-routing stream.

In [ ]:
input_path = RESULTS_DIR / "notebook22_predictive_decompression.csv"

if input_path.exists():
    stream = pd.read_csv(input_path)
    print("Loaded:", input_path)
else:
    print("Notebook 22 output not found; creating fallback synthetic compressed-routing stream.")
    rng = np.random.default_rng(43)
    N = 240
    windows = np.arange(N)
    macro_routes = rng.choice(
        ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"],
        size=N,
        p=[0.34, 0.12, 0.20, 0.18, 0.16],
    )

    macro_cgcs_score = (
        0.55
        + 0.08 * np.sin(np.linspace(0, 8 * np.pi, N))
        + rng.normal(0, 0.08, N)
    )
    macro_cgcs_score[106:145] += 0.13
    macro_cgcs_score[35:55] -= 0.12
    macro_cgcs_score[205:230] -= 0.10
    macro_cgcs_score = np.clip(macro_cgcs_score, 0.20, 0.84)

    rolling_stability = pd.Series(macro_cgcs_score).rolling(12, min_periods=1).mean()
    rolling_volatility = pd.Series(macro_cgcs_score).rolling(10, min_periods=1).std().fillna(0)
    rolling_switch_rate = (
        pd.Series(macro_routes)
        .ne(pd.Series(macro_routes).shift())
        .rolling(15, min_periods=1)
        .mean()
    )
    rolling_residual = 1 - rolling_stability + rolling_volatility
    rolling_pressure_raw = (
        0.40 * rolling_switch_rate
        + 0.35 * rolling_volatility
        + 0.25 * rolling_residual
    )
    rolling_pressure = (
        (rolling_pressure_raw - rolling_pressure_raw.min())
        / (rolling_pressure_raw.max() - rolling_pressure_raw.min())
    )

    decompression_event = ((macro_cgcs_score < 0.45) & (rolling_pressure > 0.55)).astype(int)
    future_horizon = 5
    future_decompression = (
        pd.Series(decompression_event)
        .rolling(future_horizon, min_periods=1)
        .max()
        .shift(-future_horizon)
        .fillna(0)
        .astype(int)
    )

    stream = pd.DataFrame({
        "window_id": windows,
        "macro_route": macro_routes,
        "macro_cgcs_score": macro_cgcs_score,
        "rolling_stability": rolling_stability,
        "rolling_volatility": rolling_volatility,
        "rolling_switch_rate": rolling_switch_rate,
        "rolling_residual": rolling_residual,
        "rolling_pressure": rolling_pressure,
        "decompression_event": decompression_event,
        "future_decompression": future_decompression,
    })

stream = stream.copy().reset_index(drop=True)

if "window_id" not in stream.columns:
    if "window" in stream.columns:
        stream["window_id"] = stream["window"]
    else:
        stream["window_id"] = np.arange(len(stream))

if "window" not in stream.columns:
    stream["window"] = stream["window_id"]

if "macro_route" not in stream.columns:
    stream["macro_route"] = "macro_unknown"

for c in ["macro_cgcs_score", "rolling_stability", "rolling_volatility",
          "rolling_switch_rate", "rolling_residual", "rolling_pressure"]:
    if c not in stream.columns:
        stream[c] = 0.5
    stream[c] = pd.to_numeric(stream[c], errors="coerce").fillna(0.5)

if "decompression_event" not in stream.columns:
    stream["decompression_event"] = ((stream["macro_cgcs_score"] < 0.45) & (stream["rolling_pressure"] > 0.55)).astype(int)

if "future_decompression" not in stream.columns:
    future_horizon = 5
    stream["future_decompression"] = (
        pd.Series(stream["decompression_event"])
        .rolling(future_horizon, min_periods=1)
        .max()
        .shift(-future_horizon)
        .fillna(0)
        .astype(int)
    )
else:
    future_horizon = 5

stream.head()

## Train or reuse decompression forecast

If Notebook 22 already produced a decompression probability, this notebook uses it.

Otherwise, it trains a lightweight random forest forecaster.

In [ ]:
route_encoding = {r: i for i, r in enumerate(sorted(stream["macro_route"].astype(str).unique()))}
stream["macro_route_id"] = stream["macro_route"].astype(str).map(route_encoding)

feature_cols = [
    "macro_cgcs_score",
    "rolling_stability",
    "rolling_volatility",
    "rolling_switch_rate",
    "rolling_residual",
    "rolling_pressure",
    "macro_route_id",
]

X = stream[feature_cols].copy()
y = stream["future_decompression"].astype(int)

if "decompression_probability" in stream.columns:
    stream["decompression_probability"] = pd.to_numeric(stream["decompression_probability"], errors="coerce").fillna(0.0)
    stream["forecast_decompression"] = (stream["decompression_probability"] >= 0.50).astype(int)
    if y.nunique() == 2:
        auc = roc_auc_score(y, stream["decompression_probability"])
    else:
        auc = float("nan")
    forecast_model = None
    feature_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": np.nan,
    })
else:
    stratify = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=43,
        stratify=stratify,
    )

    forecast_model = RandomForestClassifier(
        n_estimators=250,
        max_depth=6,
        min_samples_leaf=3,
        random_state=43,
    )
    forecast_model.fit(X_train, y_train)

    if len(forecast_model.classes_) == 2:
        class_one_index = list(forecast_model.classes_).index(1)
        stream["decompression_probability"] = forecast_model.predict_proba(X)[:, class_one_index]
    else:
        stream["decompression_probability"] = 0.0

    stream["forecast_decompression"] = (stream["decompression_probability"] >= 0.50).astype(int)
    auc = roc_auc_score(y, stream["decompression_probability"]) if y.nunique() == 2 else float("nan")

    feature_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": forecast_model.feature_importances_,
    }).sort_values("importance", ascending=False)

print("ROC AUC:", auc)
stream[["window_id", "future_decompression", "decompression_probability", "forecast_decompression"]].head()

## Reactive vs predictive routing policies

Reactive routing falls back only when decompression is observed.

Predictive routing acts before collapse when forecast risk is high:

- `accepted`
- `watch`
- `reroute`
- `fallback`

In [ ]:
reactive_gate = np.where(
    stream["decompression_event"] == 1,
    "fallback",
    np.where(stream["macro_cgcs_score"] >= 0.70, "accepted", "watch"),
)

predictive_gate = []
predictive_action = []

for _, row in stream.iterrows():
    risk = row["decompression_probability"]
    cgcs = row["macro_cgcs_score"]
    pressure = row["rolling_pressure"]

    if risk >= 0.70 and pressure >= 0.55:
        predictive_gate.append("fallback")
        predictive_action.append("protective_fallback")
    elif risk >= 0.50:
        predictive_gate.append("reroute")
        predictive_action.append("predictive_reroute")
    elif cgcs >= 0.70:
        predictive_gate.append("accepted")
        predictive_action.append("retain_compressed")
    else:
        predictive_gate.append("watch")
        predictive_action.append("monitor")

stream["reactive_gate"] = reactive_gate
stream["predictive_gate"] = predictive_gate
stream["predictive_action"] = predictive_action

stream["forecast_hit"] = (
    (stream["forecast_decompression"] == 1)
    & (stream["future_decompression"] == 1)
).astype(int)

stream["avoidable_fallback"] = (
    (stream["future_decompression"] == 1)
    & (stream["predictive_gate"].isin(["reroute", "fallback"]))
).astype(int)

stream[["window_id", "reactive_gate", "predictive_gate", "predictive_action", "avoidable_fallback"]].head()

## Stability and switching proxy metrics

In [ ]:
reactive_stability = np.clip(
    stream["rolling_stability"]
    - 0.22 * (stream["reactive_gate"] == "fallback").astype(float)
    - 0.10 * stream["future_decompression"],
    0,
    1,
)

predictive_stability = np.clip(
    stream["rolling_stability"]
    + 0.12 * (stream["predictive_gate"] == "reroute").astype(float)
    + 0.06 * (stream["predictive_gate"] == "accepted").astype(float)
    - 0.11 * (stream["predictive_gate"] == "fallback").astype(float)
    - 0.04 * stream["future_decompression"],
    0,
    1,
)

stream["reactive_stability"] = reactive_stability
stream["predictive_stability"] = predictive_stability
stream["stability_gain"] = predictive_stability - reactive_stability

stream["reactive_switch"] = (
    pd.Series(stream["reactive_gate"])
    .ne(pd.Series(stream["reactive_gate"]).shift())
    .astype(int)
)

stream["predictive_switch"] = (
    pd.Series(stream["predictive_gate"])
    .ne(pd.Series(stream["predictive_gate"]).shift())
    .astype(int)
)

stream["reactive_switch_rate"] = stream["reactive_switch"].rolling(15, min_periods=1).mean()
stream["predictive_switch_rate"] = stream["predictive_switch"].rolling(15, min_periods=1).mean()

stream[[
    "window_id",
    "reactive_stability",
    "predictive_stability",
    "stability_gain",
    "reactive_switch_rate",
    "predictive_switch_rate",
]].head()

## Summaries and transition matrices

In [ ]:
gate_order = ["accepted", "watch", "reroute", "fallback"]

def transition_matrix(labels, order):
    mat = pd.DataFrame(0.0, index=order, columns=order)
    labels = list(labels)
    for i in range(len(labels) - 1):
        mat.loc[labels[i], labels[i + 1]] += 1
    return mat.div(mat.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

predictive_transition_matrix = transition_matrix(stream["predictive_gate"], gate_order)
reactive_transition_matrix = transition_matrix(stream["reactive_gate"], ["accepted", "watch", "fallback"])

action_summary = (
    stream.groupby("predictive_action")
    .agg(
        windows=("window_id", "count"),
        mean_probability=("decompression_probability", "mean"),
        mean_pressure=("rolling_pressure", "mean"),
        mean_cgcs=("macro_cgcs_score", "mean"),
        mean_stability_gain=("stability_gain", "mean"),
        avoidable_fallback_rate=("avoidable_fallback", "mean"),
    )
    .reset_index()
    .sort_values("windows", ascending=False)
)

route_summary = (
    stream.groupby("macro_route")
    .agg(
        windows=("window_id", "count"),
        mean_probability=("decompression_probability", "mean"),
        mean_pressure=("rolling_pressure", "mean"),
        mean_cgcs=("macro_cgcs_score", "mean"),
        mean_stability_gain=("stability_gain", "mean"),
        predictive_reroute_rate=("predictive_action", lambda s: (s == "predictive_reroute").mean()),
        protective_fallback_rate=("predictive_action", lambda s: (s == "protective_fallback").mean()),
    )
    .reset_index()
    .sort_values("mean_probability", ascending=False)
)

summary = {
    "windows": int(len(stream)),
    "roc_auc": float(auc) if not np.isnan(auc) else None,
    "forecast_positive_windows": int(stream["forecast_decompression"].sum()),
    "actual_future_decompression_windows": int(stream["future_decompression"].sum()),
    "predictive_reroute_windows": int((stream["predictive_gate"] == "reroute").sum()),
    "reactive_fallback_windows": int((stream["reactive_gate"] == "fallback").sum()),
    "predictive_fallback_windows": int((stream["predictive_gate"] == "fallback").sum()),
    "avoidable_fallback_windows": int(stream["avoidable_fallback"].sum()),
    "mean_reactive_stability": float(stream["reactive_stability"].mean()),
    "mean_predictive_stability": float(stream["predictive_stability"].mean()),
    "mean_stability_gain": float(stream["stability_gain"].mean()),
    "mean_reactive_switch_rate": float(stream["reactive_switch_rate"].mean()),
    "mean_predictive_switch_rate": float(stream["predictive_switch_rate"].mean()),
}

summary_df = pd.DataFrame([summary])

classification_df = pd.DataFrame(
    classification_report(
        stream["future_decompression"],
        stream["forecast_decompression"],
        labels=[0, 1],
        output_dict=True,
        zero_division=0,
    )
).transpose()

summary_df

## Save result artifacts

In [ ]:
routing_csv = RESULTS_DIR / "notebook23_predictive_constraint_routing.csv"
routing_json = RESULTS_DIR / "notebook23_predictive_constraint_routing.json"
summary_csv = RESULTS_DIR / "notebook23_summary.csv"
action_summary_csv = RESULTS_DIR / "notebook23_action_summary.csv"
route_summary_csv = RESULTS_DIR / "notebook23_route_summary.csv"
feature_importance_csv = RESULTS_DIR / "notebook23_feature_importance.csv"
predictive_transition_csv = RESULTS_DIR / "notebook23_predictive_gate_transition_matrix.csv"
reactive_transition_csv = RESULTS_DIR / "notebook23_reactive_gate_transition_matrix.csv"
classification_csv = RESULTS_DIR / "notebook23_forecast_classification_report.csv"

stream.to_csv(routing_csv, index=False)
stream.to_json(routing_json, orient="records", indent=2)
summary_df.to_csv(summary_csv, index=False)
action_summary.to_csv(action_summary_csv, index=False)
route_summary.to_csv(route_summary_csv, index=False)
feature_importance.to_csv(feature_importance_csv, index=False)
predictive_transition_matrix.to_csv(predictive_transition_csv)
reactive_transition_matrix.to_csv(reactive_transition_csv)
classification_df.to_csv(classification_csv)

print("Saved:", routing_csv)
print("Saved:", routing_json)
print("Saved:", summary_csv)
print("Saved:", action_summary_csv)
print("Saved:", route_summary_csv)
print("Saved:", feature_importance_csv)
print("Saved:", predictive_transition_csv)
print("Saved:", reactive_transition_csv)
print("Saved:", classification_csv)

## Figure 1 — Forecast risk timeline

In [ ]:
risk_timeline_fig = FIGURES_DIR / "notebook23_decompression_risk_timeline.png"

plt.figure(figsize=(16, 6))
plt.plot(stream["window_id"], stream["decompression_probability"], label="forecast probability")
plt.axhline(0.50, linestyle="--", label="reroute threshold")
plt.axhline(0.70, linestyle="--", label="protective fallback threshold")
plt.title("Predictive Constraint Routing: Decompression Risk Timeline")
plt.xlabel("Window")
plt.ylabel("Forecast probability")
plt.legend()
plt.tight_layout()
plt.savefig(risk_timeline_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", risk_timeline_fig)

## Figure 2 — Reactive vs predictive gate timeline

In [ ]:
gate_timeline_fig = FIGURES_DIR / "notebook23_reactive_vs_predictive_gate_timeline.png"

state_map = {"accepted": 3, "watch": 2, "reroute": 1, "fallback": 0}
reactive_map = {"accepted": 3, "watch": 2, "fallback": 0}

plt.figure(figsize=(16, 6))
plt.step(
    stream["window_id"],
    [reactive_map[v] for v in stream["reactive_gate"]],
    where="post",
    label="reactive gate",
)
plt.step(
    stream["window_id"],
    [state_map[v] for v in stream["predictive_gate"]],
    where="post",
    linestyle="--",
    label="predictive gate",
)
plt.yticks([0, 1, 2, 3], ["fallback", "reroute", "watch", "accepted"])
plt.title("Predictive Constraint Routing: Reactive vs Predictive Gate Timeline")
plt.xlabel("Window")
plt.ylabel("Gate")
plt.legend()
plt.tight_layout()
plt.savefig(gate_timeline_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", gate_timeline_fig)

## Figure 3 — Stability comparison

In [ ]:
stability_fig = FIGURES_DIR / "notebook23_reactive_vs_predictive_stability.png"

plt.figure(figsize=(16, 6))
plt.plot(stream["window_id"], stream["reactive_stability"], label="reactive stability")
plt.plot(stream["window_id"], stream["predictive_stability"], label="predictive stability")
plt.title("Predictive Constraint Routing: Stability Before vs After Forecast Routing")
plt.xlabel("Window")
plt.ylabel("Stability score")
plt.legend()
plt.tight_layout()
plt.savefig(stability_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", stability_fig)

## Figure 4 — Switch-rate comparison

In [ ]:
switch_fig = FIGURES_DIR / "notebook23_reactive_vs_predictive_switch_rates.png"

plt.figure(figsize=(16, 6))
plt.plot(stream["window_id"], stream["reactive_switch_rate"], label="reactive switch rate")
plt.plot(stream["window_id"], stream["predictive_switch_rate"], label="predictive switch rate")
plt.title("Predictive Constraint Routing: Reactive vs Predictive Switch Rates")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.legend()
plt.tight_layout()
plt.savefig(switch_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", switch_fig)

## Figure 5 — Action counts

In [ ]:
action_counts_fig = FIGURES_DIR / "notebook23_action_counts.png"

plt.figure(figsize=(10, 6))
plt.bar(action_summary["predictive_action"], action_summary["windows"])
plt.xticks(rotation=35, ha="right")
plt.title("Predictive Constraint Routing: Action Counts")
plt.ylabel("Window count")
plt.tight_layout()
plt.savefig(action_counts_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", action_counts_fig)

## Figure 6 — Stability gain by route

In [ ]:
stability_gain_fig = FIGURES_DIR / "notebook23_stability_gain_by_macro_route.png"

plt.figure(figsize=(10, 6))
plt.bar(route_summary["macro_route"], route_summary["mean_stability_gain"])
plt.xticks(rotation=35, ha="right")
plt.title("Predictive Constraint Routing: Mean Stability Gain by Macro Route")
plt.ylabel("Mean stability gain")
plt.tight_layout()
plt.savefig(stability_gain_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", stability_gain_fig)

## Figure 7 — Predictive gate transition matrix

In [ ]:
predictive_transition_fig = FIGURES_DIR / "notebook23_predictive_gate_transition_matrix.png"

plt.figure(figsize=(7, 6))
plt.imshow(predictive_transition_matrix.values, aspect="auto")
plt.xticks(range(len(gate_order)), gate_order, rotation=35, ha="right")
plt.yticks(range(len(gate_order)), gate_order)
plt.colorbar(label="Transition probability")
plt.title("Predictive Constraint Routing: Predictive Gate Transition Matrix")
plt.xlabel("Next gate")
plt.ylabel("Current gate")
plt.tight_layout()
plt.savefig(predictive_transition_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", predictive_transition_fig)

## Figure 8 — Pressure, risk, and intervention

In [ ]:
pressure_risk_fig = FIGURES_DIR / "notebook23_pressure_risk_intervention.png"

plt.figure(figsize=(16, 6))
plt.plot(stream["window_id"], stream["rolling_pressure"], label="rolling pressure")
plt.plot(stream["window_id"], stream["decompression_probability"], label="decompression probability")

reroute_windows = stream[stream["predictive_gate"] == "reroute"]
fallback_windows = stream[stream["predictive_gate"] == "fallback"]

plt.scatter(
    reroute_windows["window_id"],
    reroute_windows["decompression_probability"],
    label="predictive reroute",
)
plt.scatter(
    fallback_windows["window_id"],
    fallback_windows["decompression_probability"],
    marker="x",
    label="protective fallback",
)
plt.title("Predictive Constraint Routing: Pressure, Risk, and Intervention")
plt.xlabel("Window")
plt.ylabel("Normalized score")
plt.legend()
plt.tight_layout()
plt.savefig(pressure_risk_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", pressure_risk_fig)

## Figure 9 — Avoided fallback summary

In [ ]:
avoided_fig = FIGURES_DIR / "notebook23_avoided_fallback_summary.png"

avoid_summary = pd.DataFrame({
    "category": ["actual future decompression", "forecast positives", "avoidable fallback windows"],
    "windows": [
        int(stream["future_decompression"].sum()),
        int(stream["forecast_decompression"].sum()),
        int(stream["avoidable_fallback"].sum()),
    ],
})

plt.figure(figsize=(10, 6))
plt.bar(avoid_summary["category"], avoid_summary["windows"])
plt.xticks(rotation=25, ha="right")
plt.title("Predictive Constraint Routing: Avoided Fallback Summary")
plt.ylabel("Window count")
plt.tight_layout()
plt.savefig(avoided_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", avoided_fig)

## Full Markdown report

In [ ]:
report_path = REPORTS_DIR / "report_23_predictive_constraint_routing.md"

# Use figures/ link style in report.
routing_csv_link = "results/notebook23_predictive_constraint_routing.csv"
routing_json_link = "results/notebook23_predictive_constraint_routing.json"
summary_csv_link = "results/notebook23_summary.csv"
action_summary_csv_link = "results/notebook23_action_summary.csv"
route_summary_csv_link = "results/notebook23_route_summary.csv"
feature_importance_csv_link = "results/notebook23_feature_importance.csv"
predictive_transition_csv_link = "results/notebook23_predictive_gate_transition_matrix.csv"
reactive_transition_csv_link = "results/notebook23_reactive_gate_transition_matrix.csv"
classification_csv_link = "results/notebook23_forecast_classification_report.csv"

risk_timeline_link = "figures/notebook23_decompression_risk_timeline.png"
gate_timeline_link = "figures/notebook23_reactive_vs_predictive_gate_timeline.png"
stability_link = "figures/notebook23_reactive_vs_predictive_stability.png"
switch_link = "figures/notebook23_reactive_vs_predictive_switch_rates.png"
action_counts_link = "figures/notebook23_action_counts.png"
stability_gain_link = "figures/notebook23_stability_gain_by_macro_route.png"
predictive_transition_link = "figures/notebook23_predictive_gate_transition_matrix.png"
pressure_risk_link = "figures/notebook23_pressure_risk_intervention.png"
avoided_link = "figures/notebook23_avoided_fallback_summary.png"

report_lines = [
    "# Report 23 — Predictive Constraint Routing",
    "",
    "This report uses decompression forecasts to route before fallback occurs.",
    "",
    "Constraint view:",
    "> predictive routing is useful when forecast risk can stabilize compressed routes before fallback becomes necessary.",
    "",
    "## Generated outputs",
    "",
    f'- Predictive constraint routing CSV: <a href="{routing_csv_link}">`{routing_csv_link}`</a>',
    f'- Predictive constraint routing JSON: <a href="{routing_json_link}">`{routing_json_link}`</a>',
    f'- Summary CSV: <a href="{summary_csv_link}">`{summary_csv_link}`</a>',
    f'- Action summary CSV: <a href="{action_summary_csv_link}">`{action_summary_csv_link}`</a>',
    f'- Route summary CSV: <a href="{route_summary_csv_link}">`{route_summary_csv_link}`</a>',
    f'- Feature importance CSV: <a href="{feature_importance_csv_link}">`{feature_importance_csv_link}`</a>',
    f'- Predictive gate transition matrix CSV: <a href="{predictive_transition_csv_link}">`{predictive_transition_csv_link}`</a>',
    f'- Reactive gate transition matrix CSV: <a href="{reactive_transition_csv_link}">`{reactive_transition_csv_link}`</a>',
    f'- Forecast classification report CSV: <a href="{classification_csv_link}">`{classification_csv_link}`</a>',
    f'- Figure: <a href="{risk_timeline_link}">`{risk_timeline_link}`</a>',
    f'- Figure: <a href="{gate_timeline_link}">`{gate_timeline_link}`</a>',
    f'- Figure: <a href="{stability_link}">`{stability_link}`</a>',
    f'- Figure: <a href="{switch_link}">`{switch_link}`</a>',
    f'- Figure: <a href="{action_counts_link}">`{action_counts_link}`</a>',
    f'- Figure: <a href="{stability_gain_link}">`{stability_gain_link}`</a>',
    f'- Figure: <a href="{predictive_transition_link}">`{predictive_transition_link}`</a>',
    f'- Figure: <a href="{pressure_risk_link}">`{pressure_risk_link}`</a>',
    f'- Figure: <a href="{avoided_link}">`{avoided_link}`</a>',
    "",
    "## Summary",
    "",
    summary_df.to_markdown(index=False),
    "",
    "## Predictive action summary",
    "",
    action_summary.to_markdown(index=False),
    "",
    "## Macro route summary",
    "",
    route_summary.to_markdown(index=False),
    "",
    "## Feature importance",
    "",
    feature_importance.to_markdown(index=False),
    "",
    "## Predictive gate transition probabilities",
    "",
    predictive_transition_matrix.to_markdown(),
    "",
    "## Reactive gate transition probabilities",
    "",
    reactive_transition_matrix.to_markdown(),
    "",
    "## Forecast classification report",
    "",
    classification_df.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Predictive routing converts high decompression probability into route actions before fallback is observed.",
    "- Predictive reroute windows act as intermediate states between watch and fallback.",
    "- Protective fallback is reserved for high-risk, high-pressure windows.",
    "- Stability gain estimates whether forecast-informed routing improves compressed-route reliability.",
    "- Avoidable fallback windows identify where forecasting can reduce reactive collapse handling.",
    "",
    "## Next step",
    "",
    "Notebook 24 can build route-memory policy evaluation:",
    "- compare routing policies across repeated trials,",
    "- estimate stability distributions,",
    "- score policy regret,",
    "- select routing policies under CGCS-style constraints.",
]

report_path.write_text("\n".join(report_lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if running in Google Colab.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook23_predictive_constraint_routing_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook23_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_23_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))